# Scoring Review with AgenticReviewer

This notebook demonstrates how to use `ScoringReviewer` from the v2 agentic API to score documents.

**Key v2 changes:** Instead of provider wrapper classes (e.g., `OpenAIProvider`), v2 uses Pydantic AI model strings like `"openai:gpt-5.4-mini"` or `"google-gla:gemini-3-flash-preview"`. This simplifies configuration and leverages Pydantic AI's built-in model support.

We'll cover:
1. Non-agentic mode (`max_iterations=1`) — a single LLM call
2. Agentic mode (default `max_iterations=20`) — the agent can loop, reflect, and use tools

## Setup and Sample Data

In [ ]:
from lattereview.agentic import ScoringReviewer, ScoringOutput
from pydantic_ai.models.test import TestModel

# Sample medical AI article abstracts
sample_items = [
    "A deep learning model was developed to detect diabetic retinopathy from fundus photographs. "
    "The model achieved 97.5% sensitivity and 93.4% specificity on a validation set of 10,000 images, "
    "outperforming board-certified ophthalmologists in a head-to-head comparison.",

    "We conducted a systematic review of 45 studies on natural language processing for clinical notes. "
    "Most studies used transformer-based architectures. Only 12 studies validated on external datasets, "
    "and none reported deployment in clinical workflows.",

    "This opinion piece argues that AI regulation in healthcare is premature. "
    "The author suggests that existing medical device frameworks are sufficient "
    "and that additional regulation would stifle innovation without improving patient safety.",
]

## Non-Agentic Scoring (Single LLM Call)

Setting `max_iterations=1` disables the agentic loop. The reviewer makes one LLM call and returns the result. This is fast and cost-effective for straightforward scoring tasks.

In [ ]:
# Using TestModel for demonstration (no API key needed)
scorer = ScoringReviewer(
    name="Relevance Scorer",
    backstory="You are an expert in medical AI research quality assessment.",
    model=TestModel(),
    scoring_task="Rate the methodological rigor of this medical AI study.",
    scoring_set=[1, 2, 3, 4, 5],
    scoring_rules="1=opinion/no data, 2=weak methods, 3=adequate, 4=strong methods, 5=exceptional rigor",
    max_iterations=1,  # Non-agentic: single LLM call
)

# Review a single item
result, cost = await scorer.review_item(sample_items[0])
print("Result:", result)
print(f"Cost: ${cost:.6f}")

In [ ]:
# Review all items at once
results, total_cost = await scorer.review_items(sample_items)

for i, r in enumerate(results):
    print(f"\nItem {i+1}:")
    print(f"  Score: {r.get('score')}")
    print(f"  Reasoning: {r.get('reasoning', 'N/A')[:100]}...")

print(f"\nTotal cost: ${total_cost:.6f}")

## Agentic Scoring (with Reflection Loop)

With the default `max_iterations=20`, the reviewer can reflect on its initial assessment, reconsider, and refine its score across multiple reasoning steps.

You can also control effort via `agentic_effort` (`"low"`, `"medium"`, `"high"`).

> **Note:** The cell below uses `TestModel` for structure demonstration. For real agentic behavior, replace with a real model string like `model="openai:gpt-5.4-mini"`.

In [ ]:
agentic_scorer = ScoringReviewer(
    name="Rigorous Scorer",
    backstory="You are a meticulous medical AI research evaluator who carefully considers study design, sample size, validation approach, and clinical relevance.",
    model=TestModel(),
    scoring_task="Rate the methodological rigor of this medical AI study.",
    scoring_set=[1, 2, 3, 4, 5],
    scoring_rules="1=opinion/no data, 2=weak methods, 3=adequate, 4=strong methods, 5=exceptional rigor",
    # max_iterations=20 is the default — agent can loop and reflect
    agentic_effort="medium",
)

result, cost = await agentic_scorer.review_item(sample_items[0])
print("Result:", result)
print(f"Cost: ${cost:.6f}")

## Using a Real Model

> **Requires API key:** Set `OPENAI_API_KEY` in your environment before running this cell.

In [ ]:
# Uncomment to run with a real model:

# real_scorer = ScoringReviewer(
#     name="Relevance Scorer",
#     backstory="You are an expert in medical AI research quality assessment.",
#     model="openai:gpt-5.4-mini",
#     scoring_task="Rate the methodological rigor of this medical AI study.",
#     scoring_set=[1, 2, 3, 4, 5],
#     scoring_rules="1=opinion/no data, 2=weak, 3=adequate, 4=strong, 5=exceptional",
#     max_iterations=1,
# )
# result, cost = await real_scorer.review_item(sample_items[0])
# print(result)

## Inspecting the Result Dict

The result dict from `review_item` contains the structured output fields from `ScoringOutput`:

In [ ]:
result, cost = await scorer.review_item(sample_items[1])

print("Keys in result dict:", list(result.keys()))
print()
for key, value in result.items():
    print(f"{key}: {value}")